##### Importação de Pacotes

In [0]:
import re
from pyspark.sql.utils import AnalysisException
from pyspark.sql.functions import to_timestamp, date_format, col

##### Diretórios nos Catálogos

In [0]:
path_bronze          = '/Volumes/01_bronze/schemas/tse/perfil_eleitorado'
path_silver          = '/Volumes/02_silver/schemas/tse/perfil_eleitorado'

##### Verifica arquivos salvos (por ANO_ELEICAO) na camada Silver

In [0]:
# Verifica quais arquivos (ANO_ELEICAO) foram carregados na camada Silver
try:
    df_silver = spark.read.format("delta").load(path_silver)

    anos_processados = [
        row["ANO_ELEICAO"]
        for row in (
            df_silver
            .select("ANO_ELEICAO")
            .distinct()
            .collect()
        )
    ]

except AnalysisException:
    anos_processados = []


In [0]:
# Validação dos arquivos na camada Silver
#anos_processados

##### Arquivos na camada Bronze

In [0]:
# Salva em uma variável todos os arquivos da camada Bronze que não estão na camada Silver
files = [f.path for f in dbutils.fs.ls(path_bronze) if f.path.endswith(".csv")]


In [0]:
# Ordena os arquivos por ANO na camada Bronze
files = sorted(
    files,
    key=lambda x: int(re.search(r'(\d{4})', x).group(1))
)


In [0]:
# Validação dos arquivos na camada Bronze na variável
#files

##### Consolida arquivos da camada Bronze 

In [0]:
# Consolida os arquivos da camada Bronze que não constam na camada Silver em um DataFrame
df_pel = (
        spark.read
        .option("header", True)
        .option("sep", ";")
        .option("encoding", "ISO-8859-1")
        .option("inferSchema", True)
        .csv(files)
    )


In [0]:
# Transforma a coluna HH_GERACAO para o formato dd/MM/yyyy HH:mm:ss
df_pel = df_pel.withColumn("HH_GERACAO", date_format(col("HH_GERACAO"),"dd/MM/yyyy HH:mm:ss"))

In [0]:
# Validação do DataFrame
#display(df_pel)

##### Salva arquivos (formato Delta) na camada Silver

In [0]:
# Salva arquivo Parquet na camada: Silver
df_pel.write \
        .format("delta") \
        .mode("append") \
        .partitionBy("ANO_ELEICAO") \
        .save(f"{path_silver}")

##### Fim